In [1]:

import numpy as np
from numpy.typing import NDArray
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.figure import Figure
from matplotlib.axes import Axes
import astropy.units as u
from astropy.io import fits
from astropy.wcs import WCS
from astropy.time import Time
from astropy.nddata import Cutout2D
from astropy.stats import sigma_clipped_stats
from astropy.coordinates import SkyCoord, EarthLocation, Angle, get_body_barycentric
from photutils.segmentation import detect_sources, SegmentationImage, SourceCatalog
from scipy.integrate import solve_ivp
from scipy.optimize import least_squares, OptimizeResult
import os

In [4]:
os.chdir("/home/odin/Code/Course-Assignments/PH556_Astrophysics/Attempt01/data/")
print("Current working directory: {0}".format(os.getcwd()))
allDataFileNames = os.listdir()
# print("Files in current working directory: {0}".format(allDataFileNames))
print(allDataFileNames)

files = [fits.open(filename) for filename in allDataFileNames]
# fileData = [file[0].data for file in files]
# fileHeaders = [file[0].header for file in files]

Current working directory: /home/odin/Code/Course-Assignments/PH556_Astrophysics/Attempt01/data
['20210721192429-359-RA.wcs.proc.fits', '20210721191920-421-RA.wcs.proc.fits', '20210713215458-181-RA.wcs.proc.fits', '20210721193447-216-RA.wcs.proc.fits', '20210721191411-345-RA.wcs.proc.fits', '20210713220516-046-RA.wcs.proc.fits', '20210721194505-080-RA.wcs.proc.fits', '20210713213912-742-RA.wcs.proc.fits', '20210713214440-179-RA.wcs.proc.fits', '20210713214949-265-RA.wcs.proc.fits', '20210713221024-980-RA.wcs.proc.fits', '20210714213919-746-RA.wcs.proc.fits', '20210713221533-914-RA.wcs.proc.fits', '20210721195014-013-RA.wcs.proc.fits', '20210714220020-444-RA.wcs.proc.fits']


In [5]:
def parseOBSDate(dateString):
    res1 = dateString.split("T")
    resMajor = res1[0].split("-")
    resMinor = res1[1].split(":")
    res = [float(x) for x in resMajor + resMinor]
    numVal = res[0]*(12*31)*24*60*60 + res[1]*31*24*60*60 + res[2]*24*60*60 + res[3]*60*60 + res[4]*60 + res[5]
    return [res, numVal]

In [6]:
datesOfObservationParsed = [parseOBSDate(file[0].header['DATE-OBS']) for file in files]
datesOfObservationNumVal = [date[1] for date in datesOfObservationParsed]
datesOfObservationDateVal = [date[0] for date in datesOfObservationParsed]
indicesForDateSortingAscending = np.argsort(datesOfObservationNumVal)

In [7]:
filesAscendingDate = [files[i] for i in indicesForDateSortingAscending]

In [8]:
bin1Files = filesAscendingDate[0:7]
bin2Files = filesAscendingDate[7:9]
bin3Files = filesAscendingDate[9:15]


(7, 2, 6, 15)